## 模式匹配处理序列

match/case 语句可以根据序列的结构与元素值进行模式匹配

match 的真正能力有两点：

- 析构（Destructuring）：模式不仅判断"长得像不像"，还会顺手把匹配到的值绑定到变量——相当于"高级解包"；
- 模式即校验：项数、字面量、类型、嵌套结构都能写进模式，检查与取值一步到位。

> 匹配对象（Subject）：match 后面的表达式，被匹配的数据；
> 模式（Pattern）：case 后面的写法，描述"要匹配什么结构"；
> 析构：匹配成功的同时把子结构绑定到变量；
> 卫语句（guard）：模式后面的 if 条件，对匹配结果二次筛选。

In [25]:
from websockets import InvalidMessage


def handle_command( message):
    message=message.split()
    match message:
        case ['BEEPER', frequency, times]:
            print(f'匹配到了 BEEPER，{frequency}，{times}')
        case ['NECK', angle]:
            print(f'匹配到了 NECK,{angle}')
        case ['LED', ident, intensity]:
            print(f'匹配到了 LED,{ident},{intensity}')
        case _:
            raise InvalidMessage(message)

handle_command("BEEPER 440 3")

匹配到了 BEEPER，440，3


## 序列模式的匹配规则：三个条件

匹配对象（Subject）与序列模式（Pattern）相匹配，需**同时满足 3 个条件**：

1. 匹配对象是一个**序列**；
2. 匹配对象与模式的**项数相等**（有 `*` 项时除外）；
3. 每个对应的项都相匹配，**包括嵌套的项**。

一句话总结：**序列模式 = 先断言"形状"（序列、项数、嵌套），再顺手"拆件"，卫语句负责形状之外的业务条件**。


In [26]:
animals=[['哈士奇','boy',2,(23.5,116.7)],['fafa','girl',3,(23.5,116.7)]]

for animal in animals:
    match animal:
        case [name,_,_,(latitude,longitude)] if latitude<=23.5:
            print(f"{name}在23.5度纬度")

哈士奇在23.5度纬度
fafa在23.5度纬度


序列模式可匹配 `collections.abc.Sequence` 的大多数实际子类（Actual Subclass）或虚拟子类（Virtual Subclass）：

| 兼容序列模式 | 不兼容（在 match 中视为"原子"值） |
|---|---|
| list、tuple、range、memoryview、array.array、collections.deque | str、bytes、bytearray |

为什么 str/bytes 被排除？→ 若 `'hello'` 能被当成 5 个字符的序列，`case [a, b, c]:` 这类模式会产生海量误匹配。所以这是**刻意设计**：把它们当作单个"原子"值，就像 987 是一个整数，而不是"数字序列"。

In [27]:
phone = '13127103609'

match tuple(phone):
    case ['1',*rest]:
        print('NA',rest)
    case ['2',*rest]:
        print('AF',rest)
    case ['3' | '4',*rest]:
        print('EU',rest)


NA ['3', '1', '2', '7', '1', '0', '3', '6', '0', '9']


三个边界要记牢：

1. **str / bytes / bytearray 是原子值**，不转换就永远匹配不上序列模式；
2. **生成器 / 迭代器不会被析构**——与解包赋值不同，模式只认 `Sequence` 一族；
3. **dict 也不是序列**，映射模式要等 ""「3.3 用模式匹配处理映射」" 才登场。

一句话总结：**序列模式只认 `collections.abc.Sequence` 一族；str/bytes 先转换（如 `tuple()`）再匹配，迭代器和 dict 都不在此列**。

### 模式里的六种写法

| 写法 | 示例 | 说明 |
|---|---|---|
| ① 通配符 | `_` | 匹配任意一项、不绑定值；**唯一可在同一模式中多次出现的"变量"** |
| ② 捕获变量 | `frequency` | 裸变量名，绑定匹配到的值 |
| ③ as 捕获 | `(lat, lon) as coord` | 把子模式匹配到的整个子结构再绑定一份 |
| ④ 字面量 / "或"模式 | `'BEEPER'`、`'3' \| '4'` | 字面量要求相等；`\|` 匹配任意一个分支 |
| ⑤ 星号模式 | `*rest` / `*_` | 捕获剩余任意多项（列表）；`*_` 只吞不存；**每个序列层级只能有一个 `*`** |
| ⑥ 类 / 类型模式 | `str(name)`、`float(lat)`、`Symbol()` | 运行时类型检查 + 捕获 |

In [28]:
s1='hello'
match tuple(s1):
    case [_,_,_,'l','o']:
        print('hello')


hello


In [29]:
v1 = ["哈士奇", "boy", 2, (23.5, 116.7)]

match v1:
    case [name, _, _, (latitude, longitude) as coord]:
        print(f"{name}在{coord[0]}度纬度，{coord[1]}度经度")
        print(coord)


哈士奇在23.5度纬度，116.7度经度
(23.5, 116.7)


In [30]:
# 测试字面量要求相等；`\|` 匹配任意一个分支

def is_apple(name):
    match name:
        case 'apple' | 'Apple':
            return True
        case _:
            return False

print(is_apple('apple'))
print(is_apple('Apple'))
print(is_apple('orange'))


True
True
False


In [31]:
s1='hello world 哈士奇'

match s1.split():
    case [*_,'哈士奇']:
        print('哈士奇在其中')
    case _:
        print('哈士奇不在其中')


哈士奇在其中


In [32]:
l1 = ["hello", 2, 3.14]

for item in l1:
    match item:
        case str(item):
            print("字符串")
        case int(item):
            print("整数")
        case float(item):
            print("浮点数")
        case _:
            print("未知类型")

字符串
整数
浮点数


> 在模式上下文中，`str(name)` **看起来像构造函数调用，实际是运行时类型检查**——要求该项是 str 实例，匹配成功才把值绑定给 name，并非把 name 转换成 str。

## 切片



In [33]:
l1='hello world'
l1[0:5]

'hello'

In [34]:
l1[6:]

'world'

步长：s[a:b:c]

`c` 是**步长**，每 c 个项取一个；负步长倒着取

In [35]:
s='bicycle'
s[::3]

'bye'

In [36]:
s[::-1]

'elcycib'

In [37]:
s[::-2]

'eccb'

### slice 对象：给切片起名字

`s[a:b:c]` 只是一种**语法糖**——真正干活的是 `slice(a, b, c)` 对象。切片参数可以直接命名，让代码自文档化。

s[2:8:2]实际上是s[slice(2, 8, 2)]

slice实际上是一个对象，可以被赋值给变量，也可以被传递给函数。

In [38]:
text='01234567890'

part=slice(2,7)
part


slice(2, 7, None)

In [39]:
text[part]

'23456'

### 多维切片与 Ellipsis（NumPy 专用）

In [40]:
import numpy as np

a=np.arange(12).reshape(3,4)
a

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

In [41]:
a[1,2]

np.int64(6)

In [42]:
a[:,1]

array([1, 5, 9])

In [43]:
a[:,::2]

array([[ 0,  2],
       [ 4,  6],
       [ 8, 10]])

### 切片赋值：可变序列的"柔性手术"

切片赋值作用于**可变序列**，可以在删除/插入的同时改变序列长度：

In [44]:
l=list(range(10))
l

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [45]:
l[2:5]=[20,30]
l

[0, 1, 20, 30, 5, 6, 7, 8, 9]

In [46]:
del l[5:7]
l

[0, 1, 20, 30, 5, 8, 9]

In [47]:
l[3::2]=[11,22]
l

[0, 1, 20, 11, 5, 22, 9]

In [48]:
l[1:1]=[100]
l

[0, 100, 1, 20, 11, 5, 22, 9]

一个陷阱——**右侧必须是可迭代对象**：

```python
>>> l[1:2] = 100             # ① 直接赋整数报错
TypeError: can only assign an iterable
>>> l[1:2] = [100]           # ② 包成列表才行
```

## 用 + 和 \* 处理序列

`+` 和 `*` 都遵循一条基本规律：

> **两者都不修改原操作数，而是构建一个全新的序列。**

In [49]:
l=[1,2,3]
l*5

[1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 2, 3]

In [50]:
'abcd'*5

'abcdabcdabcdabcdabcd'

**注意事项**：

- `+` 两侧必须是**同类型**序列（list + tuple 报 TypeError），拼接时都不修改原序列，各自按顺序拼接成新序列
- `n * 序列` 中 n 是**重复次数**（乘 0 或负数得到空序列）
- 想把 A 中的项重复添加到 B 里，**不是** `[A] * n`，而是下面 2.8.2 的坑所在

### 嵌套列表的初始化陷阱

In [52]:
board = [['_'] * 3 for i in range(3)]
board

[['_', '_', '_'], ['_', '_', '_'], ['_', '_', '_']]

In [53]:
board[1][2]='X'
board


[['_', '_', '_'], ['_', '_', 'X'], ['_', '_', '_']]

In [60]:
from nt import TMP_MAX


tmp_board = [["_"] * 3]
tmp_board

[['_', '_', '_']]

In [62]:
weird_board = tmp_board*3
weird_board


[['_', '_', '_'], ['_', '_', '_'], ['_', '_', '_']]

In [63]:
weird_board[1][2]='0'
weird_board

[['_', '_', '0'], ['_', '_', '0'], ['_', '_', '0']]

## list.sort 与内置函数 sorted

`list.sort` 方法**就地**排序列表，返回 None；内置函数 `sorted` **接受任何可迭代对象**，返回一个**新列表**。

| | `list.sort()` | `sorted(iterable)` |
|---|---|---|
| 作用对象 | 只能是 list | **任何可迭代对象**（tuple/str/set/genexp…） |
| 修改原数据 | **就地排序**，原列表被修改 | 不碰原数据，返回**新列表** |
| 返回值 | **None** | 新的 list |
| 是否可级联 | 不能（返回 None） | 能，可接在表达式后面 |

In [64]:
fruits=['grape','raspberry','apple','banana']


In [65]:
sorted(fruits)

['apple', 'banana', 'grape', 'raspberry']

In [66]:
fruits

['grape', 'raspberry', 'apple', 'banana']

In [67]:
sorted(fruits,key=len)


['grape', 'apple', 'banana', 'raspberry']

In [71]:
fruits.sort()
fruits


['apple', 'banana', 'grape', 'raspberry']

**两个关键字参数**（sort 和 sorted 都支持）：

- `reverse=True` → 降序输出（反转比较结果，算法仍是稳定排序）
- `key=单参数函数` → 对每一项调用 key 函数，**按返回值排序**（如 `key=len`、`key=str.lower`）；key 不能与 cmp 兼容混用，但一个 key 函数能干很多事

**为什么 `list.sort()` 返回 None？——约定，不是 bug**

> Python API 有条统一约定：**函数或方法就地更改对象就返回 None**（如 `random.shuffle`、`list.sort`），好让调用者明确知道"对象变了、没有新对象诞生"。而 `sorted` 是内置函数不是方法，它必须返回新对象，不受此约定限制。

**排序稳定性**：Python 的排序算法（Timsort）是**稳定的**——排序后，key 相等的两个项保持**原来的相对顺序**。例 ④ 中 `apple` 在 `grape` 前面（输入里也是这个顺序）。巧妙用法：**先用次要 key 排一遍，再用主要 key 排**，就能实现多级排序（后一次排序不破坏前一次的同 key 顺序）。

> 一句话总结：**sort 是"就地改、还 None"的方法，sorted 是"不动原数据、给新列表"的函数**；两者都吃 `reverse`/`key` 参数，且排序稳定（key 相同保持原顺序）；返回 None = 就地修改是 Python API 的通用约定。

## 当列表不适用时

list 灵活好用，但有些场景有**更好的专用类型**——要装 1000 万个浮点数时（内存/性能），要往队首频繁加项时（复杂度），用对工具事半功倍。

### array.array：装同类型机器值的紧凑数组

若一个列表只装**数字**，`array.array` 更高效——存的是**机器值**（翻译成 C 语言的类型），没有 float 对象的标头开销（呼应本章开头的 `ob_refcnt/ob_type/ob_fval`），底层是连续的 C 数组。

In [ ]:
from array import array
from random import random

floats = array("d", [random() for i in range(1000000)])  # ① 类型码 'd' = double
floats[-1]  # 用法与 list 几乎一样

0.8086071908485131

In [73]:
with open("floats.bin", "wb") as f:
    floats.tofile(f)

| 操作 | array.tofile/fromfile | 文本方式（str 转换循环） |
|---|---|---|
| 写入耗时 | **约 0.1 秒** | 约 6 秒（慢 60 倍） |
| 读取耗时 | 约 0.1 秒 | 约 3.5 秒（慢 7 倍） |

### deque：双端队列，频繁首尾操作的首选

`.append` 和 `.pop` 在列表**末尾**操作效率高（O(1)），但插入/删除**队首**（`insert(0, x)`、`pop(0)`）是 O(n)——整个列表都要挪。`collections.deque` 是**双向队列**，两端增删都是 O(1)（线程安全的 C 实现）：

In [74]:
from collections import deque

dq = deque(range(10), maxlen=10)
dq

deque([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], maxlen=10)

In [75]:
dq.rotate(3) # 向右旋转 3 个位置:末尾三项移动到队首
dq

deque([7, 8, 9, 0, 1, 2, 3, 4, 5, 6], maxlen=10)

In [76]:
dq.rotate(-3) # 向左旋转 3 个位置:队首三项移动到队尾
dq


deque([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], maxlen=10)

In [77]:
dq.appendleft(-1)
dq

deque([-1, 0, 1, 2, 3, 4, 5, 6, 7, 8], maxlen=10)

In [78]:
dq.extendleft([10,20,30])
dq

deque([30, 20, 10, -1, 0, 1, 2, 3, 4, 5], maxlen=10)

> deque 的要点：
> - **maxlen 可选**：设置后队列有界——append 满时**从对端悄悄丢弃**旧项（不报错），适合保留"最近 N 条"
> - `rotate(n)` 旋转；`appendleft`/`popleft`/`extendleft` 操作队首
> - **`append` 和 `popleft` 是原子操作**，可以放心当多线程队列用，不用加锁
> - 代价：deque 支持索引 `dq[i]`，但**中间项访问是 O(n)**——需要随机访问就别用 deque